# TrustLens — Phase D: Pretrained Deep Visual Embeddings (DINOv2) & Dense Nearest-Neighbor Search

**Multimodal Marketplace Intelligence Engine**

This notebook performs dense semantic visual indexing on the verified OLX marketplace corpus using **DINOv2-ViT-S/14** (384-dimensional embeddings) and exact cosine nearest-neighbor search.

### Strict Scientific Constraints
- **Observational candidate generation only** (Status: ).
- **No fraud scores**, no scam labels, no automated accusations.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
print("Libraries loaded successfully!")

## 1. Load Processed Parquet Data & Phase C Ground Truth
Load , , , and .

In [ ]:
df_fps = pd.read_parquet("data/olx_processed/fingerprints.parquet")
df_listings = pd.read_parquet("data/olx_processed/normalized_listings.parquet")
df_embeddings = pd.read_parquet("data/olx_processed/image_embeddings.parquet")
df_neighbors = pd.read_parquet("data/olx_processed/visual_neighbors.parquet")
df_visual_rels = pd.read_parquet("data/olx_processed/deep_visual_relationships.parquet")
df_phase_c_rels = pd.read_parquet("data/olx_processed/image_relationships.parquet")

print(f"Loaded {len(df_embeddings):,} image embeddings (dim = {len(df_embeddings.iloc[0]['embedding'])})")
print(f"Loaded {len(df_neighbors):,} nearest neighbor records")
print(f"Loaded {len(df_visual_rels):,} cross-listing visual candidate pairs (s >= 0.70)")

## 2. Cosine Similarity Distribution & Nearest-Neighbor Decay
Plot the distribution of top-10 nearest-neighbor similarities across all indexed images.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(df_neighbors["similarity"], bins=35, color="#6366f1", edgecolor="white", alpha=0.85)
ax.axvline(0.85, color="#f43f5e", linestyle="--", linewidth=1.5, label="Candidate Filter (s >= 0.85)")
ax.set_title("DINOv2 Cosine Similarity Distribution Across Top-10 Nearest Neighbors", fontweight="bold")
ax.set_xlabel("Cosine Similarity")
ax.set_ylabel("Pair Frequency")
ax.legend()
plt.show()

## 3. Validation Against Phase C Ground Truth
Examine how DINOv2 evaluates known Phase C exact SHA-256 duplicate pairs ( = 0$) and pHash candidate pairs ( \le 10$).

In [ ]:
mid_to_vec = {r["media_id"]: np.array(r["embedding"], dtype=np.float32) for _, r in df_embeddings.iterrows()}

exact_sims = []
phash_sims = []

for _, r in df_phase_c_rels.iterrows():
    m_a, m_b = r["media_a_id"], r["media_b_id"]
    if m_a in mid_to_vec and m_b in mid_to_vec:
        sim = float(np.dot(mid_to_vec[m_a], mid_to_vec[m_b]))
        if r["relationship_type"] == "EXACT_FILE_REUSE":
            exact_sims.append(sim)
        elif r["relationship_type"] == "PERCEPTUAL_SIMILARITY_CANDIDATE":
            phash_sims.append(sim)

print(f"Phase C Exact SHA-256 (N={len(exact_sims)}): Mean DINO Sim = {np.mean(exact_sims):.6f}")
print(f"Phase C Perceptual pHash (N={len(phash_sims)}): Mean DINO Sim = {np.mean(phash_sims):.4f}")

## 4. Multi-Threshold Sensitivity Analysis
Evaluating candidate counts across varying similarity thresholds ( \ge 0.70$ to zsh.95$).

In [ ]:
thresholds = [0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
cross_nn = df_neighbors[~df_neighbors["is_same_listing"]]
counts = [int((cross_nn["similarity"] >= t).sum()) for t in thresholds]

df_thresh = pd.DataFrame({"Threshold": [f"s >= {t:.2f}" for t in thresholds], "Candidate Pairs": counts})
display(df_thresh)

## 5. Cross-City Candidate Analysis & Price Spreads
Examine visual candidates that span distinct Indian cities.

In [ ]:
meta_dict = df_listings.set_index("listing_id").to_dict(orient="index")
cross_city_pairs = []

for _, r in df_visual_rels.iterrows():
    l_a = meta_dict.get(r["listing_a_id"], {})
    l_b = meta_dict.get(r["listing_b_id"], {})
    c_a, c_b = l_a.get("city"), l_b.get("city")
    if isinstance(c_a, str) and isinstance(c_b, str) and c_a != c_b and c_a.lower() != "nan" and c_b.lower() != "nan":
        p_a, p_b = l_a.get("price_amount"), l_b.get("price_amount")
        diff = abs(p_a - p_b) if (p_a and p_b) else None
        diff_pct = (diff / max(p_a, p_b) * 100) if (p_a and p_b and max(p_a, p_b) > 0) else None
        cross_city_pairs.append({
            "Listing A": r["listing_a_id"],
            "Listing B": r["listing_b_id"],
            "City A": c_a,
            "City B": c_b,
            "Model A": l_a.get("model", "Unknown"),
            "Model B": l_b.get("model", "Unknown"),
            "Price A": p_a,
            "Price B": p_b,
            "Price Diff (%)": diff_pct,
            "Similarity": r["dino_similarity"],
        })

df_cc = pd.DataFrame(cross_city_pairs)
print(f"Total cross-city candidate relationships: {len(df_cc):,}")
display(df_cc.sort_values(by="Similarity", ascending=False).head(10))

## 6. Summary of Phase D Findings
1. **100% Exact Ground Truth Concordance**: DINOv2 achieves $pprox 1.0000$ mean cosine similarity on known SHA-256 exact matches.
2. **High pHash Alignment**: pHash candidates ( \le 10$) score a mean of zsh.9003$ similarity in DINO space.
3. **Rich Feature Space Discovery**: DINOv2 retrieves cross-listing candidate pairs across varying angles, crops, and compression artifacts.